In [ ]:
from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation

from diffusers.utils import load_image, make_image_grid

from controlnet_aux import CannyDetector, MidasDetector
from controlnet_aux.util import resize_image

import torch
import pathlib
import numpy as np
import cv2
import random

from IPython.display import display
import PIL.Image as Image

In [ ]:
def resize(image: Image.Image):
    return Image.fromarray(resize_image(np.array(image, dtype=np.uint8), 512))

# Images

## Object

In [ ]:
root_path = pathlib.Path("/home/leafying/data/UAV/DUT_Anti_UAV/detection/images")
image_paths = sorted(root_path.rglob("*.jpg"))

targets = [
    image_paths[1],
    image_paths[11],
    image_paths[12],
    image_paths[18],
    image_paths[24],
]

object_images = [resize(load_image(image_path.as_posix())) for image_path in targets]
make_image_grid(
    [resize(object_image) for object_image in object_images],
    rows=1,
    cols=len(object_images),
)

## Background

In [ ]:
root_path = pathlib.Path(
    "/home/leafying/data/Weather/Image2Weather/dataset"
)
image_paths = sorted(root_path.rglob("*.jpg"))

generator = np.random.default_rng(seed=2026)
background_images = [
    resize(load_image(image_path.as_posix()))
    for image_path in generator.choice(image_paths, size=5, replace=False)
]
make_image_grid(
    [resize(background_image) for background_image in background_images],
    rows=1,
    cols=len(background_images),
)

# Canny

In [ ]:
canny = CannyDetector()

## Object

In [ ]:
object_canny_images = [canny(object_image, low_threshold=100, high_threshold=200) for object_image in object_images]
make_image_grid(object_canny_images, rows=1, cols=len(object_canny_images))

## Background

In [ ]:
background_canny_images = [canny(background_image, low_threshold=100, high_threshold=200) for background_image in background_images]
make_image_grid(background_canny_images, rows=1, cols=len(background_canny_images))

# Depth

In [ ]:
midas = MidasDetector.from_pretrained("lllyasviel/Annotators").to("cuda")
object_depth_images = [midas(object_image) for object_image in object_images]
make_image_grid(object_depth_images, rows=1, cols=len(object_depth_images))

In [ ]:
midas = MidasDetector.from_pretrained("lllyasviel/Annotators").to("cuda")
background_depth_images = [midas(background_image) for background_image in background_images]
make_image_grid(background_depth_images, rows=1, cols=len(background_depth_images))

# Segment

In [ ]:
ADE20K_SEM_SEG_CATEGORIES = [
    "wall",
    "building",
    "sky",
    "floor",
    "tree",
    "ceiling",
    "road, route",
    "bed",
    "window ",
    "grass",
    "cabinet",
    "sidewalk, pavement",
    "person",
    "earth, ground",
    "door",
    "table",
    "mountain, mount",
    "plant",
    "curtain",
    "chair",
    "car",
    "water",
    "painting, picture",
    "sofa",
    "shelf",
    "house",
    "sea",
    "mirror",
    "rug",
    "field",
    "armchair",
    "seat",
    "fence",
    "desk",
    "rock, stone",
    "wardrobe, closet, press",
    "lamp",
    "tub",
    "rail",
    "cushion",
    "base, pedestal, stand",
    "box",
    "column, pillar",
    "signboard, sign",
    "chest of drawers, chest, bureau, dresser",
    "counter",
    "sand",
    "sink",
    "skyscraper",
    "fireplace",
    "refrigerator, icebox",
    "grandstand, covered stand",
    "path",
    "stairs",
    "runway",
    "case, display case, showcase, vitrine",
    "pool table, billiard table, snooker table",
    "pillow",
    "screen door, screen",
    "stairway, staircase",
    "river",
    "bridge, span",
    "bookcase",
    "blind, screen",
    "coffee table",
    "toilet, can, commode, crapper, pot, potty, stool, throne",
    "flower",
    "book",
    "hill",
    "bench",
    "countertop",
    "stove",
    "palm, palm tree",
    "kitchen island",
    "computer",
    "swivel chair",
    "boat",
    "bar",
    "arcade machine",
    "hovel, hut, hutch, shack, shanty",
    "bus",
    "towel",
    "light",
    "truck",
    "tower",
    "chandelier",
    "awning, sunshade, sunblind",
    "street lamp",
    "booth",
    "tv",
    "plane",
    "dirt track",
    "clothes",
    "pole",
    "land, ground, soil",
    "bannister, banister, balustrade, balusters, handrail",
    "escalator, moving staircase, moving stairway",
    "ottoman, pouf, pouffe, puff, hassock",
    "bottle",
    "buffet, counter, sideboard",
    "poster, posting, placard, notice, bill, card",
    "stage",
    "van",
    "ship",
    "fountain",
    "conveyer belt, conveyor belt, conveyer, conveyor, transporter",
    "canopy",
    "washer, automatic washer, washing machine",
    "plaything, toy",
    "pool",
    "stool",
    "barrel, cask",
    "basket, handbasket",
    "falls",
    "tent",
    "bag",
    "minibike, motorbike",
    "cradle",
    "oven",
    "ball",
    "food, solid food",
    "step, stair",
    "tank, storage tank",
    "trade name",
    "microwave",
    "pot",
    "animal",
    "bicycle",
    "lake",
    "dishwasher",
    "screen",
    "blanket, cover",
    "sculpture",
    "hood, exhaust hood",
    "sconce",
    "vase",
    "traffic light",
    "tray",
    "trash can",
    "fan",
    "pier",
    "crt screen",
    "plate",
    "monitor",
    "bulletin board",
    "shower",
    "radiator",
    "glass, drinking glass",
    "clock",
    "flag",  # noqa
]

In [ ]:
from controlnet_aux.util import ade_palette

processor = OneFormerProcessor.from_pretrained("shi-labs/oneformer_ade20k_swin_large")
model = OneFormerForUniversalSegmentation.from_pretrained(
    "shi-labs/oneformer_ade20k_swin_large"
).to("cuda")

palette = [
    {"name": ADE20K_SEM_SEG_CATEGORIES[i], "color": color}
    for i, color in enumerate(ade_palette())
]
background_segment_images = []
for background_image, background_depth_image in zip(
    background_images, background_depth_images
):
    inputs = processor(
        background_image, task_inputs=["semantic"], return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)

    seg = (
        processor.post_process_semantic_segmentation(
            outputs, target_sizes=[background_depth_image.size[::-1]]
        )[0]
        .cpu()
        .numpy()
    )

    color_seg = np.zeros(
        (seg.shape[0], seg.shape[1], 3), dtype=np.uint8
    )  # height, width, 3
    for label, info in enumerate(palette):
        # if info["name"] in ("plane",):
        #     continue

        color = info["color"]
        color_seg[seg == label, :] = color

    background_segment_image = Image.fromarray(color_seg)
    background_segment_images.append(background_segment_image)
make_image_grid(background_segment_images, rows=1, cols=len(background_segment_images))

# Inference

In [ ]:
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    StableDiffusionControlNetImg2ImgPipeline,
    UniPCMultistepScheduler,
    DPMSolverMultistepScheduler,
    KDPM2DiscreteScheduler,
    KDPM2AncestralDiscreteScheduler,
)
from compel import Compel

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"
controlnet_models = [
    "lllyasviel/control_v11p_sd15_canny",
    "lllyasviel/control_v11f1p_sd15_depth",
    "lllyasviel/control_v11p_sd15_canny",
    "lllyasviel/control_v11p_sd15_seg",
]

controlnets = [
    ControlNetModel.from_pretrained(
        controlnet_model,
        variant="fp16",
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
    for controlnet_model in controlnet_models
]
pipeline: StableDiffusionControlNetPipeline = (
    StableDiffusionControlNetPipeline.from_pretrained(
        model_id,
        controlnet=controlnets,
        variant="fp16",
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
)
# pipeline: StableDiffusionControlNetImg2ImgPipeline = (
#     StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
#         model_id,
#         controlnet=controlnets,
#         torch_dtype=torch.float16,
#         use_safetensors=True,
#     )
# )
# pipeline.scheduler = UniPCMultistepScheduler.from_config(pipeline.scheduler.config)

# DPM++ 2M Karras
# pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config, use_karras_sigmas=True)

# DPM++ 2M SDE Karras
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
    pipeline.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++"
)

# DPM2 Karras
# pipeline.scheduler = KDPM2DiscreteScheduler.from_config(
#     pipeline.scheduler.config, use_karras_sigmas=True
# )

# DPM2 a Karras
# pipeline.scheduler = KDPM2AncestralDiscreteScheduler.from_config(
#     pipeline.scheduler.config, use_karras_sigmas=True
# )

compel_proc = Compel(tokenizer=pipeline.tokenizer, text_encoder=pipeline.text_encoder)

pipeline.enable_model_cpu_offload()

In [ ]:
seed = 0

prompt_prefix = "A photograph of "
prompt_suffix = ", photorealistic, vivid, high resolution, 8k, highly detailed, Canon R6 Mark II, 35 mm lens."

prompt = f"{prompt_prefix}one drone flying in rainy weather{prompt_suffix}"
negative_prompt = "lowres, sketches, paintings"

for object_image, object_canny_image, object_depth_image in zip(object_images, object_canny_images, object_depth_images):
    generator = torch.Generator().manual_seed(seed)
    for (
        background_image,
        background_canny_image,
        background_depth_image,
        background_segment_image,
    ) in zip(
        background_images,
        background_canny_images,
        background_depth_images,
        background_segment_images,
    ):
        object_canny_image = object_canny_image.resize(
            background_canny_image.size, resample=Image.Resampling.BILINEAR
        )
        object_depth_image = object_depth_image.resize(
            background_depth_image.size, resample=Image.Resampling.BILINEAR
        ) 
        display(
            make_image_grid(
                [
                    object_canny_image,
                    object_depth_image,
                    background_canny_image,
                    background_depth_image,
                    background_segment_image,
                    object_image,
                    background_image,
                ],
                rows=1,
                cols=7,
            )
        )
        output = pipeline(
            prompt_embeds=compel_proc(prompt),
            negative_prompt_embeds=compel_proc(negative_prompt),
            # image=object_image,
            image=[
                object_canny_image,
                object_depth_image,
                background_canny_image,
                background_depth_image,
                background_segment_image,
                # background_depth_image,
                # background_segment_image,
            ],
            controlnet_conditioning_scale=[0.8, 0.8, 0.5, 0.5, 0.5],
            generator=generator,
            num_inference_steps=50,
            # guess_mode=True,
            # guidance_scale=3.0,
            # strength=0.9,
        ).images[0]
        display(output)